# Evaluation Summary

This notebook summarizes the evaluation results of the trained models.

## 1. Import Libraries

In [ ]:
import os
import yaml
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler

## 2. Define Paths and Load Configurations

In [ ]:
# Define the parent directory for experiments
output_parent_dir = os.path.abspath(os.path.expanduser(
    '/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/tool-wear-domain-adaption/experiments/conv_n_vs_2'
))

# Load configs and model paths from all experiments
experiment_data = []
config_dir = os.path.join(output_parent_dir, 'configs')
model_dir = os.path.join(output_parent_dir, 'models')

if os.path.exists(model_dir) and os.path.exists(config_dir):
    # Create a dictionary mapping timestamps to config file paths
    config_files = {
        f.split('config_')[-1].replace('.yaml', ''): os.path.join(config_dir, f)
        for f in os.listdir(config_dir) if f.endswith('.yaml')
    }
    
    # Find all model files and match them with configs
    model_files = [f for f in os.listdir(model_dir) if f.endswith('.keras')]
    for model_filename in model_files:
        model_path = os.path.join(model_dir, model_filename)
        # Extract timestamp from filename, e.g., 1d_conv_model_20251009_090843.keras
        timestamp = model_filename.split('model_')[-1].replace('.keras', '')
        
        if timestamp in config_files:
            config_path = config_files[timestamp]
            with open(config_path, 'r') as f:
                config = yaml.safe_load(f)
            
            experiment_data.append({
                'config_path': config_path,
                'model_path': model_path,
                'config': config,
                'train_caseIDs': config.get('train_caseIDs', []),
                'test_caseIDs': config.get('test_caseIDs', [])
            })

# Create a DataFrame to display the experiment configurations
if experiment_data:
    experiments_df = pd.DataFrame([{
        'Train Cases': d['train_caseIDs'],
        'Test Cases': d['test_caseIDs'],
        'Model Path': os.path.basename(d['model_path']),
        'Config Path': os.path.basename(d['config_path'])
    } for d in experiment_data])
else:
    experiments_df = pd.DataFrame()

# Sort by train and test cases for consistent order
if not experiments_df.empty:
    # Convert list to tuple for sorting
    experiments_df['Train Cases'] = experiments_df['Train Cases'].apply(tuple)
    experiments_df['Test Cases'] = experiments_df['Test Cases'].apply(tuple)
    experiments_df = experiments_df.sort_values(by=['Train Cases', 'Test Cases']).reset_index(drop=True)

experiments_df

## 3. Load and Prepare Test Data

In [ ]:
def load_data(input_path):
    return pd.read_parquet(input_path)

def set_wear_th(data_df, wear_th, wear_column_name):
    data_df['wear_class'] = np.where(data_df[wear_column_name] > wear_th, 1, 0)
    return data_df

def reformat_data(data_df, signal_columns, label_column):
    sequence_data = [
        np.stack([row[col] for col in signal_columns], axis=-1)
        for _, row in tqdm.tqdm(data_df.iterrows(), total=len(data_df), desc='Extracting sequences')
    ]
    labels = data_df[label_column].to_numpy()
    return sequence_data, labels

def rectangular_sequence_data(sequence_data, signal_length, pooling_type='mean'):
    num_samples = len(sequence_data)
    if num_samples == 0:
        return np.array([])
    num_signals = sequence_data[0].shape[1]
    rectangular = np.zeros((num_samples, signal_length, num_signals), dtype=sequence_data[0].dtype)
    for i, seq in enumerate(tqdm.tqdm(sequence_data, desc='Processing sequences')):
        seq_len = seq.shape[0]
        if seq_len > signal_length:
            # Downsample using pooling
            pool_size = int(np.ceil(seq_len / signal_length))
            if pooling_type == 'mean':
                pooled_seq = np.array([np.mean(seq[i:i+pool_size], axis=0) for i in range(0, seq_len, pool_size)])
            elif pooling_type == 'max':
                pooled_seq = np.array([np.max(seq[i:i+pool_size], axis=0) for i in range(0, seq_len, pool_size)])
            else:
                raise ValueError(f"Unknown pooling type: {pooling_type}")
            rectangular[i, :min(len(pooled_seq), signal_length)] = pooled_seq[:signal_length]
        else:
            # Pad with zeros
            rectangular[i, :seq_len] = seq
    return rectangular

def normalize_data(X_train, X_test):
    X_train_np = np.array(X_train, dtype=np.float32)
    X_test_np = np.array(X_test, dtype=np.float32)
    
    nsamples, nx, ny = X_train_np.shape
    X_train_2d = X_train_np.reshape((nsamples * nx, ny))

    scaler = StandardScaler()
    scaler.fit(X_train_2d)

    X_train_scaled_2d = scaler.transform(X_train_2d)
    X_train_scaled = X_train_scaled_2d.reshape(nsamples, nx, ny)

    nsamples_test, nx_test, ny_test = X_test_np.shape
    X_test_2d = X_test_np.reshape((nsamples_test * nx_test, ny_test))
    X_test_scaled_2d = scaler.transform(X_test_2d)
    X_test_scaled = X_test_scaled_2d.reshape(nsamples_test, nx_test, ny_test)
    
    return X_train_scaled, X_test_scaled, scaler

def prepare_test_data(config):
    # Use relative path for data to be more portable
    data_path = os.path.join(os.path.dirname(os.getcwd()), 'nasa_notebooks', config['inputPath'])
    data_df = load_data(data_path)
    data_df = set_wear_th(data_df, config['wearTH'], config['wearColumnName'])
    
    train_df = data_df[data_df['CaseID'].isin(config['train_caseIDs'])].copy()
    test_df = data_df[data_df['CaseID'].isin(config['test_caseIDs'])].copy()

    # Prepare training data to fit the scaler
    sequence_train_raw, _ = reformat_data(train_df, config['signalColumns'], config['labelColumn'])
    sequence_train = rectangular_sequence_data(sequence_train_raw, config['signal_length'], config['pooling_type'])

    # Prepare test data
    sequence_test_raw, labels_test = reformat_data(test_df, config['signalColumns'], config['labelColumn'])
    sequence_test = rectangular_sequence_data(sequence_test_raw, config['signal_length'], config['pooling_type'])
    
    # Explicitly set num_classes=2 to handle cases where test set has only one class
    labels_test_cat = to_categorical(labels_test, num_classes=2)

    # Normalize test data using a scaler fitted on the training data
    _, X_test_scaled, _ = normalize_data(sequence_train, sequence_test)
    
    return X_test_scaled, labels_test_cat

# Example of preparing data for the first experiment
if experiment_data:
    first_exp_config = experiment_data[0]['config']
    X_test_example, y_test_example = prepare_test_data(first_exp_config)
    print(f"Shape of prepared test data (X): {X_test_example.shape}")
    print(f"Shape of prepared test labels (y): {y_test_example.shape}")
else:
    print("No experiment data found.")

## 4. Evaluate Models and Collect Results

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
from IPython.display import display, Markdown

results = []

for exp_data in tqdm.tqdm(experiment_data, desc="Evaluating models"):
    # Load model from the directory path
    model = tf.keras.models.load_model(exp_data['model_path'])
    
    # Prepare data for this specific experiment's configuration
    X_test, y_test = prepare_test_data(exp_data['config'])
    
    # Evaluate model
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    
    # Get predictions
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)
    
    # Calculate confusion matrix
    cm = confusion_matrix(y_true_classes, y_pred_classes)
    
    # Calculate precision, recall, and f1-score using scikit-learn for robustness
    # zero_division=0 sets the metric to 0 if there are no true positives.
    precision = precision_score(y_true_classes, y_pred_classes, zero_division=0)
    recall = recall_score(y_true_classes, y_pred_classes, zero_division=0)
    f1 = f1_score(y_true_classes, y_pred_classes, zero_division=0)
    
    results.append({
        'train_caseIDs': exp_data['train_caseIDs'],
        'test_caseIDs': exp_data['test_caseIDs'],
        'accuracy': accuracy,
        'loss': loss,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'confusion_matrix': cm,
        'y_true': y_true_classes,
        'y_pred': y_pred_classes
    })

# Create a DataFrame to display the results
results_df = pd.DataFrame([{
    'Train Cases': r['train_caseIDs'],
    'Test Cases': r['test_caseIDs'],
    'Accuracy': r['accuracy'],
    'Precision': r['precision'],
    'Recall': r['recall'],
    'F1-Score': r['f1_score'],
    'Loss': r['loss']
} for r in results])

# Sort by train and test cases for consistent order
if not results_df.empty:
    # Convert list to tuple for sorting
    results_df['Train Cases'] = results_df['Train Cases'].apply(lambda x: tuple(x) if isinstance(x, list) else x)
    results_df['Test Cases'] = results_df['Test Cases'].apply(lambda x: tuple(x) if isinstance(x, list) else x)
    results_df = results_df.sort_values(by=['Train Cases', 'Test Cases']).reset_index(drop=True)

display(Markdown("### Overall Performance Metrics"))
display(results_df)

## 5. Visualize Confusion Matrices

In [ ]:
def plot_confusion_matrices(results):
    num_results = len(results)
    if num_results == 0:
        print("No results to display.")
        return
    
    # Determine grid size
    cols = 3
    rows = (num_results + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    axes = axes.flatten() # Flatten to 1D array for easy iteration

    for i, result in enumerate(results):
        disp = ConfusionMatrixDisplay(confusion_matrix=result['confusion_matrix'])
        disp.plot(cmap=plt.cm.Blues, ax=axes[i])
        title = f"Train: {result['train_caseIDs']}\\nTest: {result['test_caseIDs']}\\nAcc: {result['accuracy']:.2f}"
        axes[i].set_title(title)

    # Hide unused subplots
    for i in range(num_results, len(axes)):
        axes[i].set_visible(False)

    plt.tight_layout()
    plt.show()

plot_confusion_matrices(results)

## 6. Accuracy Heatmap

In [ ]:
def plot_accuracy_heatmap(results_df):
    if results_df.empty:
        print("No results to display.")
        return

    # The 'Train Cases' and 'Test Cases' are tuples, convert them to strings for plotting
    results_df['Train Cases Str'] = results_df['Train Cases'].astype(str)
    results_df['Test Cases Str'] = results_df['Test Cases'].astype(str)

    # Create a pivot table for the heatmap
    try:
        # Use pivot_table to handle duplicate entries by averaging them
        heatmap_data = results_df.pivot_table(index='Train Cases Str', columns='Test Cases Str', values='Accuracy', aggfunc='mean')
        
        plt.figure(figsize=(12, 8))
        sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="viridis")
        plt.title('Model Accuracy Heatmap (Mean Accuracy)')
        plt.xlabel('Test Case IDs')
        plt.ylabel('Train Case IDs')
        plt.show()
    except Exception as e:
        print(f"Could not create heatmap. Error: {e}")
        print("Data might not be in the right format for pivoting.")


plot_accuracy_heatmap(results_df)

## 7. Bar Chart of Accuracy per Experiment

In [ ]:
def plot_metrics_bar_chart(results_df):
    if results_df.empty:
        print("No results to display.")
        return

    # Create a descriptive label for each experiment
    results_df['Experiment'] = results_df.apply(
        lambda row: f"Train: {row['Train Cases']}\\nTest: {row['Test Cases']}", 
        axis=1
    )

    # Melt the DataFrame to have metrics in a single column for hue
    metrics_df = results_df.melt(
        id_vars=['Experiment'], 
        value_vars=['Accuracy', 'Precision', 'Recall', 'F1-Score'],
        var_name='Metric', 
        value_name='Score'
    )

    plt.figure(figsize=(15, 10))
    ax = sns.barplot(x='Score', y='Experiment', hue='Metric', data=metrics_df, orient='h')
    
    plt.title('Model Performance Metrics for Each Experiment')
    plt.xlabel('Score')
    plt.ylabel('Experiment Configuration')
    plt.xlim(0, 1.15) # Extend x-axis for legend and labels
    plt.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')

    # Add the score labels on each bar
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f', label_type='edge', padding=2)

    plt.tight_layout()
    plt.show()

plot_metrics_bar_chart(results_df)

In [ ]:
from IPython.display import display, Markdown

### 8. Analysis of Zero-Score Metrics

# Filter for experiments where precision or recall is 0
zero_metric_df = results_df[(results_df['Precision'] == 0) | (results_df['Recall'] == 0)].copy()

if not zero_metric_df.empty:
    display(Markdown("#### Experiments with Zero Precision or Recall"))
    display(Markdown(
        "The following experiments resulted in a precision or recall of 0. "
        "This typically happens when the model fails to predict any positive cases (zero precision) "
        "or when there are no positive cases in the test set (zero recall)."
    ))
    analysis_data = []
    for i, row in zero_metric_df.iterrows():
        # Find the corresponding full result dictionary
        # Convert list to tuple for comparison to match DataFrame dtypes
        res = next((r for r in results if tuple(r['train_caseIDs']) == row['Train Cases'] and tuple(r['test_caseIDs']) == row['Test Cases']), None)
        
        if res:
            y_true = res['y_true']
            y_pred = res['y_pred']
            
            # Count positive samples in true and predicted labels
            true_positives_count = np.sum(y_true)
            pred_positives_count = np.sum(y_pred)
            
            analysis_data.append({
                'Train Cases': row['Train Cases'],
                'Test Cases': row['Test Cases'],
                'Accuracy': row['Accuracy'],
                'Precision': row['Precision'],
                'Recall': row['Recall'],
                'True Positives in Test Set': f"{true_positives_count} / {len(y_true)}",
                'Predicted Positives': f"{pred_positives_count} / {len(y_pred)}"
            })
        
    analysis_df = pd.DataFrame(analysis_data)
    display(analysis_df)
else:
    display(Markdown("All experiments have non-zero precision and recall."))